# ST-SCKM Parameter Tuning

Grid search over spatial weight, temporal weight, KNN size, and penalty strength.

In [ ]:
import pandas as pd
from stsckm import STSCKM, add_default_features, evaluate_labels, load_sample_wildfire, standardize_features

In [ ]:
df = load_sample_wildfire()
df = add_default_features(df)
X_spatial, _ = standardize_features(df, ["x_proj", "y_proj"])
X_temporal, _ = standardize_features(df, ["time_days"])
X_eval = pd.concat([pd.DataFrame(X_spatial), pd.DataFrame(X_temporal)], axis=1).to_numpy()

In [ ]:
results = []
for ws in [0.5, 1.0, 1.5]:
    for wt in [0.5, 1.0, 1.5]:
        for lam in [0.0, 0.5, 1.0]:
            for k in [3, 5, 8]:
                model = STSCKM(n_clusters=4, spatial_weight=ws, temporal_weight=wt, lambda_spatial=lam, n_neighbors=k, random_state=42)
                labels = model.fit_predict(X_spatial, X_temporal)
                metrics = evaluate_labels(X_eval, labels)
                results.append({"spatial_weight": ws, "temporal_weight": wt, "lambda_spatial": lam, "n_neighbors": k, **metrics})

tuning = pd.DataFrame(results).sort_values("silhouette", ascending=False)
tuning.head(10)